# Monthly pipeline demo

Notebook version of the monthly pipeline. Pipeline logic is imported from the Python files in this folder.

In [4]:
year = 2026
month = 8
ym = f"{year}{month:02d}"

## Setup and imports

In [ ]:
# Run on Google Colab (Uncomment this cell and comment next cell)

# # Install dependencies
# !pip install arxiv polars sentence-transformers pyyaml scikit-learn spacy tqdm

# # Upload the who pipeline/ folder. Then set the folder as the working directory:
# %cd /content/ai-trends

# from pathlib import Path
# import sys
# import os
# import polars as pl

# # Adds the pipeline's package directory (parent of current notebook) to the search path
# sys.path.append(os.path.abspath('..'))
# # Import scripts from the package
# from pipeline.classify import classify, cosine_similarity
# from pipeline.embed import load_model, load_or_create
# from pipeline.fetch import fetch_month
# from pipeline.io import load_config, load_nodes, load_timeseries, write_timeseries
# from pipeline.keywords import extract_keywords
# from pipeline.paths import PipelinePaths
# from pipeline.stats import update_timeseries

# # For simplicity, all config and data files stay in the currently directory.
# # Remember to upload the data files needed to the current directory.
# # All generated files also stay in current directory. 
# # Download the updated timeseries.json and generated outputs afterward.
# paths = PipelinePaths(
#     root=Path.cwd(),
#     config=Path("settings.yml"),
#     metadata=Path("metadata.json"),
#     timeseries=Path("timeseries.json"),
#     arxiv_dir=Path("."),
#     embeddings_dir=Path("."),
#     classified_dir=Path("."),
# )
# paths.ensure_output_dirs()

In [ ]:
from pathlib import Path
import sys
import os
import polars as pl

# Adds the pipeline's package directory (parent of current notebook) to the search path
sys.path.append(os.path.abspath('..'))
# Import scripts from the package
from pipeline.classify import classify, cosine_similarity
from pipeline.embed import load_model, load_or_create
from pipeline.fetch import fetch_month
from pipeline.io import load_config, load_nodes, load_timeseries, write_timeseries
from pipeline.keywords import extract_keywords
from pipeline.paths import PipelinePaths
from pipeline.stats import update_timeseries

paths = PipelinePaths.default()
paths.ensure_output_dirs()


In [ ]:
# Configurations
config = load_config(paths.config)
categories = config["arxiv"]["categories"]
confidence_threshold = config["classification"]["confidence_threshold"]
t1_gap = config["classification"]["t1_gap"]
top_n_candidates= config["classification"]["top_n_candidates"]
embedding_model = config["embedding"]["model"]
embedding_device = config["embedding"].get("device")
batch_size = config["embedding"]["batch_size"]

# Metadata
l1_nodes = load_nodes(paths.metadata, level=1)
l2_nodes = load_nodes(paths.metadata, level=2)
l2_node_ids = list(l2_nodes)
node_texts = [f"Query: {node['N']}: {node['D']}" for node in l2_nodes.values()]

## Fetch monthly papers

In [ ]:
# This calls the live arXiv API and overwrites data/arxiv_data/{ym}.parquet.
paper_path = fetch_month(year, month, categories, paths.arxiv_dir)


## Load embedding model

In [6]:
model = load_model(embedding_model, embedding_device)

## Load or generate embeddings

In [7]:
# Load fetched data
arxiv_data = pl.read_parquet(paths.arxiv_dir / f"{ym}.parquet")

# Load or generate embeddings
node_embeddings = load_or_create(paths.embeddings_dir / "nodes.npy", model, node_texts, batch_size)
abstract_embeddings = load_or_create(
    paths.embeddings_dir / f"{ym}_abstracts.npy",
    model,
    arxiv_data["abstract"].to_list(),
    batch_size,
)

Loaded embeddings: /home/manho/repos/ai-trends/data/checkpoints/embeddings/nodes.npy
Loaded embeddings: /home/manho/repos/ai-trends/data/checkpoints/embeddings/202608_abstracts.npy


## Classify papers

In [8]:
similarities = cosine_similarity(abstract_embeddings, node_embeddings)
classified, ambiguous = classify(
    arxiv_data,
    similarities,
    l2_node_ids,
    confidence_threshold,
    t1_gap,
    top_n_candidates,
)

classified_path = paths.classified_dir / f"{ym}_classified.parquet"
pl.DataFrame(classified).write_parquet(classified_path)
pl.DataFrame(ambiguous).write_parquet(paths.classified_dir / f"{ym}_ambiguous.parquet")
print(f"Classified: {len(classified)} | Ambiguous: {len(ambiguous)}")

Classified: 3443 | Ambiguous: 9762


## Extract keywords

In [ ]:
labeled_data = pl.read_parquet(classified_path).join(
    arxiv_data.select(["arxiv_id", "abstract"]),
    on="arxiv_id",
    how="left",
)
keywords = extract_keywords(labeled_data, l2_node_ids, config)


## Update the time series

In [ ]:
timeseries = update_timeseries(
    year,
    month,
    load_timeseries(paths.timeseries),
    labeled_data,
    keywords,
    l1_nodes,
    l2_nodes,
)
# write_timeseries(paths.timeseries, timeseries)
# print(f"Updated timeseries for {year}-{month:02d}")